# TA-06 Pemodelan Sistem Dinamik: Dinamika Suhu Ruangan Ber-AC
Reyna Noernaila (301240033)

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
C = 150000.0      # kapasitas kalor efektif (J/K)
UA = 40.0         # konduktansi termal dinding (W/K)
T_out = 32.0      # suhu luar (C), konstan
Q_int = 200.0     # beban panas internal (W), gangguan
Q_max = 600.0     # kapasitas pendinginan AC (W)
dt = 10.0         # langkah waktu (s)
t_end = 6 * 3600  # lama simulasi (s)

tau = C / UA
t = np.arange(0, t_end + dt, dt)
th = t / 3600     # waktu dalam jam (untuk grafik)
print("Konstanta waktu tau:", tau, "detik =", tau / 60, "menit")
print("Jumlah langkah waktu:", len(t))

Konstanta waktu tau: 3750.0 detik = 62.5 menit
Jumlah langkah waktu: 2161


In [3]:
def T_steady(u):
    return T_out + (Q_int - Q_max * u) / UA

def simulate(u_func, T0):
    T = np.zeros_like(t)
    u = np.zeros_like(t)
    T[0] = T0
    u[0] = u_func(t[0])
    for i in range(1, len(t)):
        dTdt = (UA * (T_out - T[i-1]) + Q_int - Q_max * u[i-1]) / C
        T[i] = T[i-1] + dTdt * dt      # metode Euler
        u[i] = u_func(t[i])
    return T, u

for u in (0.0, 0.5, 1.0):
    print("u =", u, "-> T steady state =", T_steady(u), "C")

u = 0.0 -> T steady state = 37.0 C
u = 0.5 -> T steady state = 29.5 C
u = 1.0 -> T steady state = 22.0 C


In [4]:
# Euler manual: dt = 600 s, u = 0, T(0) = 32 C
T_ss0 = T_steady(0.0)
T_n = 32.0
print("n    t(s)    T_n   UA(Tout-T)   dT/dt     T_n+1   T_analitik(t_n)")
for n in range(6):
    aliran = UA * (T_out - T_n)
    dTdt = (aliran + Q_int) / C
    T_next = T_n + dTdt * 600
    T_ana = T_ss0 + (32.0 - T_ss0) * np.exp(-n * 600 / tau)
    print(f"{n}  {n*600:5d}  {T_n:7.4f}  {aliran:9.2f}  {dTdt:.6f}  {T_next:7.4f}  {T_ana:7.4f}")
    T_n = T_next

n    t(s)    T_n   UA(Tout-T)   dT/dt     T_n+1   T_analitik(t_n)
0      0  32.0000       0.00  0.001333  32.8000  32.0000
1    600  32.8000     -32.00  0.001120  33.4720  32.7393
2   1200  33.4720     -58.88  0.000941  34.0365  33.3693
3   1800  34.0365     -81.46  0.000790  34.5106  33.9061
4   2400  34.5106    -100.43  0.000664  34.9089  34.3635
5   3000  34.9089    -116.36  0.000558  35.2435  34.7534


In [5]:
# verifikasi program (dt = 10 s) terhadap solusi analitik
T_num, _ = simulate(lambda s: 0.0, 32.0)
T_ana = T_ss0 + (32.0 - T_ss0) * np.exp(-t / tau)
print("Galat maksimum Euler vs analitik:", np.max(np.abs(T_num - T_ana)), "C")
print("T(tau) simulasi:", T_num[int(tau / dt)], "| T(tau) analitik:", T_ana[int(tau / dt)])

Galat maksimum Euler vs analitik: 0.0024552582795251965 C
T(tau) simulasi: 35.163058052422315 | T(tau) analitik: 35.16060279414279


In [6]:
plt.figure(figsize=(8, 4.5))
for u, warna in ((0.0, 'tab:red'), (0.5, 'tab:blue'), (1.0, 'tab:green')):
    T, _ = simulate(lambda s, u=u: u, 32.0)
    plt.plot(th, T, color=warna, label=f'u = {u}')
    plt.axhline(T_steady(u), color=warna, linestyle='--', linewidth=0.8)
    print(f"u = {u}: T pada jam ke-6 = {T[-1]:.3f} C | T_ss = {T_steady(u):.1f} C")
plt.xlabel('Waktu (jam)')
plt.ylabel('Suhu ruangan T (C)')
plt.title('Simulasi steady state suhu ruangan (T(0) = 32 C)')
plt.legend()
plt.grid(True)
plt.savefig('grafik_steady_state.png', dpi=200)
plt.show()

u = 0.0: T pada jam ke-6 = 36.984 C | T_ss = 37.0 C
u = 0.5: T pada jam ke-6 = 29.508 C | T_ss = 29.5 C
u = 1.0: T pada jam ke-6 = 22.031 C | T_ss = 22.0 C


In [7]:
t_on = 3600.0   # sinyal aktif pada t = 1 jam
u_step    = lambda s: 0.5 if s >= t_on else 0.0
u_impulse = lambda s: 1.0 if t_on <= s < t_on + 300 else 0.0
u_ramp    = lambda s: min(max((s - t_on) / 3600 * 0.25, 0.0), 1.0)
u_sin     = lambda s: (0.5 + 0.4 * np.sin(2 * np.pi * (s - t_on) / 7200)
                       if s >= t_on else 0.0)

T0 = T_steady(0.0)   # kondisi awal = steady state tanpa AC
skenario = [('Step', u_step), ('Impulse (pulsa 5 menit)', u_impulse),
            ('Ramp', u_ramp), ('Sinusoida', u_sin)]

fig, ax = plt.subplots(4, 2, figsize=(11, 11))
for k, (nama, f) in enumerate(skenario):
    T, u = simulate(f, T0)
    ax[k, 0].plot(th, u * 100, 'r--')
    ax[k, 0].set_ylabel('u(t) (%)')
    ax[k, 0].set_title('Sinyal kontrol: ' + nama)
    ax[k, 1].plot(th, T, 'b')
    ax[k, 1].set_ylabel('T(t) (C)')
    ax[k, 1].set_title('Respon suhu: ' + nama)
    ax[k, 0].grid(True); ax[k, 1].grid(True)
    print(f"{nama}: T min = {T.min():.2f} C | T jam ke-6 = {T[-1]:.2f} C")
    if nama == 'Step':
        print("   T pada t_on + tau =", round(T[int((t_on + tau) / dt)], 2), "C")
ax[3, 0].set_xlabel('Waktu (jam)'); ax[3, 1].set_xlabel('Waktu (jam)')
plt.tight_layout()
plt.savefig('grafik_sinyal_kontrol.png', dpi=200)
plt.show()

Step: T min = 29.56 C | T jam ke-6 = 29.56 C
   T pada t_on + tau = 32.26 C
Impulse (pulsa 5 menit): T min = 35.85 C | T jam ke-6 = 36.99 C
Ramp: T min = 23.46 C | T jam ke-6 = 23.46 C
Sinusoida: T min = 27.80 C | T jam ke-6 = 27.87 C
